# Module 01 — Lab: Prompting in practice

You will:
1. Refactor a vague prompt into a structured one and measure the change.
2. Add few-shot examples to a classifier and compare zero-shot vs few-shot.
3. Force strict JSON output with prefilling.
4. Compare 'just answer' vs extended thinking on a reasoning task.

In [ ]:
import os, json
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv('../.env')
client = Anthropic()
MODEL = os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-6')

def ask(messages, system=None, max_tokens=512, **kw):
    return client.messages.create(
        model=MODEL, max_tokens=max_tokens, system=system or [], messages=messages, **kw
    )

## 1. Bad prompt vs structured prompt

We'll summarize a paragraph two ways and read both outputs.

In [ ]:
DOC = '''The Q3 release shipped on schedule despite the staging outage on Sept 14.
Adoption of the new dashboard reached 42% of weekly actives by month's end, beating
the 30% target. Two known regressions remain: search latency over 3s for users with
>10k records, and a CSS bug on Safari 17. Engineering allocated 2 sprints to fix both.'''

bad = ask([{'role':'user','content': f'summarize this: {DOC}'}])
print('--- bad ---')
print(bad.content[0].text)

good_system = '''You are a release-notes summarizer.
Output exactly three bullets: (1) what shipped, (2) one positive metric, (3) one risk.
Each bullet under 20 words. No marketing language.'''

good = ask(
    [{'role':'user','content': f'<document>\n{DOC}\n</document>'}],
    system=good_system,
)
print('\n--- good ---')
print(good.content[0].text)

## 2. Few-shot lift on a classifier

We score five inputs with zero-shot and three-shot. Eyeball the difference in tone consistency.

In [ ]:
REVIEWS = [
    'Absolute masterpiece. Cried twice.',
    'It was fine. I watched it.',
    'Pacing was off and the third act unraveled.',
    'Mid.',
    'Worth seeing for the cinematography alone.',
]

ZERO_SHOT = 'Classify the review as positive, negative, or neutral. Reply with one word only.'

FEW_SHOT = '''Classify the review as positive, negative, or neutral.
Reply with one word only.

<examples>
<example><input>Loved every minute.</input><output>positive</output></example>
<example><input>Total waste of time.</input><output>negative</output></example>
<example><input>It exists.</input><output>neutral</output></example>
</examples>'''

def classify(system, text):
    r = ask([{'role':'user','content': f'<review>{text}</review>'}], system=system, max_tokens=8)
    return r.content[0].text.strip().lower()

print(f"{'review':<55} {'zero':<10} {'few':<10}")
for r in REVIEWS:
    z = classify(ZERO_SHOT, r)
    f = classify(FEW_SHOT, r)
    print(f'{r[:53]:<55} {z:<10} {f:<10}')

## 3. Force JSON with prefilling

We assert the response parses on the first try.

In [ ]:
messages = [
    {'role': 'user', 'content': 'Give me populations for Tokyo, London, Lagos. '
                                'Return JSON {"cities":[{"name":..., "population":int}]}.'},
    {'role': 'assistant', 'content': '{'},  # prefill
]
r = ask(messages, max_tokens=256)
raw = '{' + r.content[0].text
print('raw:', raw)
print('parsed:', json.loads(raw))

## 4. Extended thinking on a reasoning task

Same math problem, two configs. Thinking blocks are returned but separate from final answer.

In [ ]:
PROBLEM = ('A train leaves City A at 9:00 going 80 km/h. Another leaves City B '
           '(420 km from A, same line) at 10:00 going 100 km/h toward A. '
           'At what clock time do they meet? Give just the clock time.')

plain = client.messages.create(
    model=MODEL, max_tokens=256,
    messages=[{'role':'user','content': PROBLEM}],
)
print('plain:', plain.content[-1].text)

thinking = client.messages.create(
    model=MODEL, max_tokens=4096,
    thinking={'type': 'enabled', 'budget_tokens': 4000},
    messages=[{'role':'user','content': PROBLEM}],
)
for b in thinking.content:
    print(f"[{b.type}]", getattr(b, 'thinking', None) or getattr(b, 'text', ''))

---
Now try the exercises in `exercises.md`.